# Biohub · v14 (CPU, CV only) — inference with our self-trained baseline (UNet detector + node transformer + ILP)

**Inputs**: competition data + `biohub-prep-bundle-v2` Output (wheels + repo) + the **v13 training Output**
(`weights/unet_transformer/split_0/edge_predictor_best.pth`). GPU T4 x2, Internet off.

1. `RUN_CV=True`: scores a few post-processing settings on the 12 hold-out clips (never trained on) with our
   official-style scorer — the same 12 clips / scorer as every earlier CV, so numbers are comparable (v12 = 0.8175).
2. `FINAL_CONFIG` is applied to the hidden test set → `submission.csv`.


In [ ]:
# ------------------------------------------------------------------ 0. settings
import os, sys, glob, time, json, subprocess, importlib, shutil
os.environ["TQDM_DISABLE"] = "1"
from pathlib import Path

RUN_CV            = True
N_CV_SAMPLES      = 12
MAX_TEST_SAMPLES  = 0               # None = all test clips; 1 = quick debug; 0 = CV only, no submission (e.g. CPU run)
DET_TTA           = False            # 4-flip test-time augmentation for detection (4x U-Net cost; set False on CPU)
WEIGHTS_PREFER    = "edge_predictor_best.pth"   # or "edge_predictor_last.pth"
CV_CONFIGS = {   # CPU run: one config, no TTA
    "ilp_t95_notta": dict(det_threshold=0.95, use_ilp=True, ilp_disappearance_weight=1.4, ilp_division_weight=1.0),
}
FINAL_CONFIG      = "ilp_t95_notta"
TIME_BUDGET_SEC   = 11.0 * 3600
MODEL_PREFER      = "ctc"           # (Trackastra folder preference; unused here, kept for the shared locate cell)

KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
OUT_DIR      = Path(os.environ.get("KAGGLE_OUTPUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print("input root:", KAGGLE_INPUT, "| output:", OUT_DIR)


In [ ]:
# ------------------------------------------------------------------ 1. locate inputs (fast: no recursive glob over the 87 GB dataset)
COMP_SLUG = "biohub-cell-tracking-during-development"

def _find_comp_dir():
    for c in [KAGGLE_INPUT / "competitions" / COMP_SLUG, KAGGLE_INPUT / COMP_SLUG]:
        if (c / "test").is_dir():
            return c
    for c in KAGGLE_INPUT.glob("*/"):              # one level deep only
        if (c / "test").is_dir() and (c / "sample_submission.csv").exists():
            return c
    for c in KAGGLE_INPUT.glob("*/*/"):
        if (c / "test").is_dir() and (c / "sample_submission.csv").exists():
            return c
    return None

def _walk_pruned(root, max_depth=8):
    """os.walk that never descends into the competition data or zarr/geff stores."""
    root = str(root)
    for dp, dns, fns in os.walk(root):
        depth = dp[len(root):].count(os.sep)
        dns[:] = [d for d in dns if not d.endswith((".zarr", ".geff")) and d != COMP_SLUG and d != "competitions"]
        if depth >= max_depth:
            dns[:] = []
        yield Path(dp), fns

COMP_DIR = _find_comp_dir()
MODEL_DIRS, PKG_DIR, CACHE_DIRS, REPO_DIRS, CKPT_DIRS = [], None, [], [], []
for dp, fns in _walk_pruned(KAGGLE_INPUT):
    if "model.pt" in fns and "train_config.yaml" in fns:
        MODEL_DIRS.append(dp)
    if PKG_DIR is None and any(f.startswith("trackastra-") and f.endswith(".whl") for f in fns):
        PKG_DIR = dp
    if dp.name == "feat_cache" and any(f.endswith(".npz") for f in fns):
        CACHE_DIRS.append(dp)
    if dp.name == "tracking_cellmot" and "io.py" in fns:            # official baseline repo (prep-bundle v2: repo/src/tracking_cellmot)
        REPO_DIRS.append(dp.parent.parent)
    if "edge_predictor_best.pth" in fns or "last.pth" in fns:        # baseline checkpoints (ours or the repo's layout)
        CKPT_DIRS.append(dp)
# model folder preference: MODEL_PREFER (e.g. "ctc_ft" = fine-tuned) first, then the pretrained "ctc"
_pref = globals().get("MODEL_PREFER", "ctc_ft")
MODEL_DIRS.sort(key=lambda p: (p.name != _pref, p.name != "ctc", str(p)))
MODEL_DIR = MODEL_DIRS[0] if MODEL_DIRS else None
PRETRAINED_DIR = next((p for p in MODEL_DIRS if p.name == "ctc"), MODEL_DIR)

print("COMP_DIR :", COMP_DIR)
print("MODEL_DIR:", MODEL_DIR, "(all found:", [str(p) for p in MODEL_DIRS], ")")
print("PKG_DIR  :", PKG_DIR)
print("CACHE_DIRS:", [str(p) for p in CACHE_DIRS])
REPO_DIR = REPO_DIRS[0] if REPO_DIRS else None
print("REPO_DIR :", REPO_DIR, "| CKPT_DIRS:", [str(p) for p in CKPT_DIRS])
print("inputs attached:", [p.name for p in KAGGLE_INPUT.iterdir()] if KAGGLE_INPUT.exists() else "n/a")
assert COMP_DIR is not None, "competition data not attached (Add Input -> Competitions -> " + COMP_SLUG + ")"
if MODEL_DIR is None or PKG_DIR is None:
    print("\n" + "!" * 78)
    print("!! Trackastra bundle is NOT attached. Add Input -> Your Work -> 'biohub-prep-bundle' notebook output")
    print("!! (it holds the offline wheels for zarr/trackastra and the pretrained Trackastra weights).")
    print("!" * 78 + "\n")

TEST_ZARRS = sorted(p for p in (COMP_DIR / "test").iterdir() if p.name.endswith(".zarr"))
n_train = sum(1 for p in (COMP_DIR / "train").iterdir() if p.name.endswith(".zarr")) if (COMP_DIR / "train").is_dir() else -1
print(f"{len(TEST_ZARRS)} test samples (hidden set is ~ train size = {n_train}):", [p.name for p in TEST_ZARRS[:4]], "...")
import torch
if not torch.cuda.is_available():
    print("!! No GPU: set Settings -> Accelerator -> GPU (T4/P100).")


In [ ]:
# ------------------------------------------------------------------ 2. offline install (no internet)
NEEDED = {  # import name -> wheel project name
    "trackastra": "trackastra", "geff": "geff", "geff_spec": "geff_spec", "zarr": "zarr",
    "numcodecs": "numcodecs", "donfig": "donfig", "google_crc32c": "google_crc32c",
    "edt": "edt", "lz4": "lz4", "imagecodecs": "imagecodecs", "chardet": "chardet",
    "platformdirs": "platformdirs", "psutil": "psutil", "dask": "dask", "numba": "numba",
    "skimage": "scikit_image", "yaml": "pyyaml", "joblib": "joblib", "tqdm": "tqdm",
    "pandas": "pandas", "scipy": "scipy", "torch": "torch", "torchvision": "torchvision",
    "typing_extensions": "typing_extensions", "pydantic": "pydantic", "requests": "requests",
    "motile": "motile", "ilpy": "ilpy", "pyscipopt": "pyscipopt", "structsvm": "structsvm",  # optional: enables mode="ilp"
    # official-baseline (tracking_cellmot) stack, present only in prep-bundle v2
    "tracksdata": "tracksdata", "polars": "polars", "rustworkx": "rustworkx", "bidict": "bidict", "psygnal": "psygnal",
    "blosc2": "blosc2", "rstar_python": "rstar_python", "sqlalchemy": "sqlalchemy", "pyarrow": "pyarrow", "rich": "rich",
    "ndindex": "ndindex", "msgpack": "msgpack", "numexpr": "numexpr",
}

def _importable(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

def _pip(args):
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-warn-script-location", "-q"] + args
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
    return r.returncode == 0

def _installed(mod):
    """present on disk? (importlib.util.find_spec does NOT import the module -> nothing stale gets loaded)"""
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

TRACKASTRA_OK = False
if PKG_DIR is not None:
    import importlib.util, importlib.metadata as im
    from packaging.version import Version
    # step A0: Kaggle ships OLD versions of some libs that tracksdata needs newer (polars < 1.36 lacks pl.Float16).
    # Upgrade from the bundle BEFORE anything imports them (re-importing a native lib after upgrade can crash the kernel).
    UPGRADE = ["polars", "polars_runtime_32", "rustworkx", "psygnal", "bidict", "blosc2", "rstar_python", "numcodecs"]
    ups = []
    for proj in UPGRADE:
        w = sorted(glob.glob(str(PKG_DIR / f"{proj}-*.whl")))
        if not w:
            continue
        ver = Path(w[-1]).name.split("-")[1]
        try:
            inst = im.version(proj.replace("_", "-"))
        except Exception:
            inst = None
        if inst is None or Version(inst) < Version(ver):
            ups.append(w[-1]); print(f"    upgrade {proj}: {inst} -> {ver}")
    if ups:
        _pip(["--no-deps", "--upgrade"] + ups); importlib.invalidate_caches()
    # step A: only the missing projects (checked without importing), no dependency resolution (keeps Kaggle torch/numpy intact)
    missing = [proj for mod, proj in NEEDED.items() if not _installed(mod)]
    print("missing before install:", missing)
    wheels = []
    for proj in missing:
        w = sorted(glob.glob(str(PKG_DIR / f"{proj}-*.whl")))
        wheels += w[:1] if w else []
    if wheels:
        _pip(["--no-deps"] + wheels); importlib.invalidate_caches()
    still = [proj for mod, proj in NEEDED.items() if not _importable(mod)]      # now really import to validate
    print("missing after step A:", still)
    if still:
        # step B: let pip resolve, but pin already-installed heavy libs so they are not replaced
        cons = OUT_DIR / "constraints.txt"
        pins = []
        for proj in ["torch", "torchvision", "numpy", "scipy", "pandas", "scikit-image", "numba", "llvmlite"]:
            try: pins.append(f"{proj}=={im.version(proj)}")
            except Exception: pass
        cons.write_text("\n".join(pins))
        extra = ["tracksdata"] if glob.glob(str(PKG_DIR / "tracksdata-*.whl")) else []
        _pip(["--find-links", str(PKG_DIR), "-c", str(cons), "trackastra", "zarr", "geff"] + extra)
        importlib.invalidate_caches()
    TRACKASTRA_OK = _importable("trackastra") and _importable("zarr")
else:
    TRACKASTRA_OK = _importable("trackastra") and _importable("zarr")

import numpy as np, torch
print("python", sys.version.split()[0], "| numpy", np.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(), "| trackastra importable:", TRACKASTRA_OK)
assert _importable("zarr"), ("zarr is not installed and no wheel bundle found -> attach the prep-bundle notebook output "
    "(it ships zarr/numcodecs/trackastra wheels for Python 3.12)")


In [ ]:
# ------------------------------------------------------------------ 3. code + model
import base64
CODE_DIR = OUT_DIR / "code"; CODE_DIR.mkdir(exist_ok=True)
for _name, _b in [("pipeline.py", "..."),
                  ("cellmot_train.py", "IiIiCkJpb2h1YiDigJQgdHJhaW4gdGhlIG9yZ2FuaXNlcnMnIG9mZmljaWFsIGJhc2VsaW5lIChUZW1wb3JhbFVOZXQzRCBkZXRlY3RvciArIFNpbXBsZU5vZGVUcmFuc2Zvcm1lciBsaW5rZXIsCnJlcG8gcm95ZXJsYWIva2FnZ2xlLWNlbGwtdHJhY2tpbmctY29tcGV0aXRpb24pIG9uIEthZ2dsZSwgd2l0aCB0aGUgdGhpbmdzIHRoZSBvcmlnaW5hbCBzY3JpcHQgbGFja3M6CgogICogZnVsbC1tb2RlbCByZXN1bWUgKG1vZGVsICsgb3B0aW1pemVyICsgZXBvY2ggY291bnRlcikgc28gYSAxMiBoIEthZ2dsZSBzZXNzaW9uIGNhbiBjb250aW51ZSB0aGUgcHJldmlvdXMgb25lCiAgKiBhIHdhbGwtY2xvY2sgYnVkZ2V0OiBzdG9wIGNsZWFubHkgYW5kIHNhdmUgYmVmb3JlIEthZ2dsZSBraWxscyB0aGUga2VybmVsCiAgKiBsYXN0LnB0aCArIGJlc3QucHRoIGV2ZXJ5IGVwb2NoLCB3cml0dGVuIGluIHRoZSByZXBvJ3Mgb3duIGxheW91dCAod2VpZ2h0cy88bWV0aG9kPi9zcGxpdF8wL2VkZ2VfcHJlZGljdG9yX2Jlc3QucHRoCiAgICArIGNvbmZpZy5qc29uKSBzbyB0aGUgcmVwbydzIHByZWRpY3Qgc2NyaXB0IOKAlCBhbmQgb3VyIG93biBpbmZlcmVuY2Ugbm90ZWJvb2sg4oCUIGNhbiBsb2FkIHRoZW0KICAqIGEgZml4ZWQgdmFsaWRhdGlvbiBzcGxpdCA9IHRoZSBzYW1lIDEyIGhvbGQtb3V0IGNsaXBzIHVzZWQgYnkgYWxsIG91ciBlYXJsaWVyIENWcwogICogZmFsbGJhY2tzIHNvIHRoZSBjb2RlIGFsc28gcnVucyBvbiBDUFUgLyBvbiBkYXRhIHdpdGhvdXQgcHJlY29tcHV0ZWQgcXVhbnRpbGVzIChsb2NhbCB0ZXN0cyBvbiBzeW50aGV0aWMgZGF0YSkKCkV2ZXJ5dGhpbmcgbW9kZWwvbG9zcy9kYXRhIHJlbGF0ZWQgaXMgaW1wb3J0ZWQgdW5jaGFuZ2VkIGZyb20gdGhlIHJlcG8gKHNjcmlwdHMvdHJhaW5fdW5ldF90cmFuc2Zvcm1lci5weSkuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCgpfVCA9IE5vbmUgICMgdGhlIHJlcG8ncyB0cmFpbiBtb2R1bGUsIHNldCBieSBzZXR1cF9yZXBvKCkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSByZXBvIGltcG9ydCArIHNtYWxsIGNvbXBhdGliaWxpdHkgcGF0Y2hlcwpkZWYgc2V0dXBfcmVwbyhyZXBvX2RpcjogUGF0aCk6CiAgICAiIiJQdXQgdGhlIHJlcG8gb24gc3lzLnBhdGgsIGltcG9ydCBpdHMgdHJhaW4gbW9kdWxlLCBwYXRjaCBDUFUvcXVhbnRpbGUgZmFsbGJhY2tzLiBSZXR1cm5zIHRoZSBtb2R1bGUuIiIiCiAgICBnbG9iYWwgX1QKICAgIGltcG9ydCB0b3JjaAogICAgcmVwb19kaXIgPSBQYXRoKHJlcG9fZGlyKQogICAgZm9yIHAgaW4gKHJlcG9fZGlyIC8gInNyYyIsIHJlcG9fZGlyIC8gInNjcmlwdHMiKToKICAgICAgICBpZiBzdHIocCkgbm90IGluIHN5cy5wYXRoOgogICAgICAgICAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKHApKQogICAgaW1wb3J0IHRyYWNraW5nX2NlbGxtb3QuaW8gYXMgY2lvCiAgICBpbXBvcnQgdHJhaW5fdW5ldF90cmFuc2Zvcm1lciBhcyBUCgogICAgX29yaWdfb3BlbiA9IGNpby5vcGVuX2RhdGFzZXQKCiAgICBkZWYgb3Blbl9kYXRhc2V0X3dpdGhfcXVhbnRpbGVzKGRzX3BhdGgsICphLCAqKmspOgogICAgICAgIGRzID0gX29yaWdfb3Blbihkc19wYXRoLCAqYSwgKiprKQogICAgICAgIGlmICIwLjAwMSIgbm90IGluIGRzLnF1YW50aWxlcyBvciAiMC45OTkiIG5vdCBpbiBkcy5xdWFudGlsZXM6CiAgICAgICAgICAgIGltcG9ydCB6YXJyCiAgICAgICAgICAgIGFyciA9IHphcnIub3Blbl9ncm91cChzdHIoZHMuemFycl9wYXRoKSwgbW9kZT0iciIpWyIwIl0KICAgICAgICAgICAgVF8gPSBhcnIuc2hhcGVbMF0KICAgICAgICAgICAgaWR4ID0gbnAudW5pcXVlKG5wLmxpbnNwYWNlKDAsIFRfIC0gMSwgbWluKDUsIFRfKSkucm91bmQoKS5hc3R5cGUoaW50KSkKICAgICAgICAgICAgc2FtcGxlID0gbnAuY29uY2F0ZW5hdGUoW25wLmFzYXJyYXkoYXJyW2ludCh0KV0pLnJhdmVsKClbOjoxM10gZm9yIHQgaW4gaWR4XSkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgICAgIGxvLCBoaSA9IG5wLnF1YW50aWxlKHNhbXBsZSwgWzAuMDAxLCAwLjk5OV0pCiAgICAgICAgICAgIGRzLnF1YW50aWxlcyA9IGRpY3QoZHMucXVhbnRpbGVzKTsgZHMucXVhbnRpbGVzLnVwZGF0ZSh7IjAuMDAxIjogZmxvYXQobG8pLCAiMC45OTkiOiBmbG9hdChoaSl9KQogICAgICAgIHJldHVybiBkcwoKICAgIGNpby5vcGVuX2RhdGFzZXQgPSBvcGVuX2RhdGFzZXRfd2l0aF9xdWFudGlsZXMKICAgIFQub3Blbl9kYXRhc2V0ID0gb3Blbl9kYXRhc2V0X3dpdGhfcXVhbnRpbGVzICAgICAgICAgICMgdGhlIHNjcmlwdCBpbXBvcnRlZCB0aGUgbmFtZSBkaXJlY3RseQogICAgaWYgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSA9IGxhbWJkYSAqYSwgKiprOiBOb25lICAgICAgICMgdHJhaW5fZXBvY2goKSBjYWxscyBpdCB1bmNvbmRpdGlvbmFsbHkKICAgIGZyb20gdHFkbSBpbXBvcnQgdHFkbSBhcyBfdHFkbQogICAgVC50cWRtID0gbGFtYmRhICphLCAqKms6IF90cWRtKCphLCAqKnsqKmssICJkaXNhYmxlIjogVHJ1ZX0pICAgIyB0aGUgc2NyaXB0IGZvcmNlcyBkaXNhYmxlPUZhbHNlIC0+IHNpbGVuY2UgcGVyLWl0ZXJhdGlvbiBiYXJzCgogICAgIyBDYXAgdGhlIG51bWJlciBvZiBkZXRlY3Rpb25zIHBlciBmcmFtZSBmZWQgdG8gdGhlIGVkZ2UgdHJhbnNmb3JtZXIuIEVhcmx5IGluIHRyYWluaW5nIHRoZSBkZXRlY3RvciBmaXJlcwogICAgIyBhbG1vc3QgZXZlcnl3aGVyZSAodGhvdXNhbmRzIG9mIHBlYWtzL2ZyYW1lKSBhbmQgdGhlIGNyb3NzLWF0dGVudGlvbiBtYXRyaXggKG5fc3JjIHggbl90Z3QpIGJsb3dzIHRoZSBHUFUKICAgICMgKEthZ2dsZSBUNDogIlRyaWVkIHRvIGFsbG9jYXRlIDYgR2lCIiBpbiBzb2Z0bWF4KS4gS2VlcGluZyBvbmx5IHRoZSB0b3AtSyBwZWFrcyBieSBsb2dpdCBpcyBzYWZlOiByZWFsIGNlbGxzCiAgICAjIGFyZSA8IDcwMC9mcmFtZSBhbmQgR1Qgbm9kZXMgYXJlIHRoZSBzdHJvbmdlc3QgcGVha3MuIEltcGxlbWVudGVkIGJ5IGxvd2VyaW5nIGV2ZXJ5IHZveGVsIGJlbG93IHRoZSBLLXRoIHBlYWsKICAgICMgbG9naXQgdG8gKHRocmVzaG9sZCAtIDEpIG9uIGEgY2xvbmUsIHNvIHRoZSBvcmlnaW5hbCBmdW5jdGlvbidzIHBlYWsgbG9naWMgaXMgdW50b3VjaGVkOyB0aGUgZGV0ZWN0aW9uIGxvc3MKICAgICMgc3RpbGwgdXNlcyB0aGUgdW5tb2RpZmllZCBsb2dpdHMuCiAgICBpbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCiAgICBfb3JpZ19kbSA9IFQuZGV0ZWN0X2FuZF9tYXRjaAoKICAgIGRlZiBkZXRlY3RfYW5kX21hdGNoX2NhcHBlZChkZXRfbG9naXRzLCBndF9jb29yZHMsIG1hc2ssIGltYWdlX3NoYXBlLCBkZXRfdGhyZXNob2xkPTAuMywgcG9vbF9rZXJuZWxfdW09NS4wLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9tYXRjaF9kaXN0YW5jZT01LjAsIHZveGVsX3NpemU9Tm9uZSwgZnJhbWVfaW5kZXg9MCwgd2luZG93X3NpemU9Tm9uZSk6CiAgICAgICAgY2FwID0gQ0FQWyJtYXhfZGV0X3Blcl9mcmFtZSJdCiAgICAgICAgaWYgY2FwOgogICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgIGlmIHZveGVsX3NpemUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAga2VybiA9IHR1cGxlKG1heCgxLCBrIGlmIGsgJSAyID09IDEgZWxzZSBrICsgMSkgZm9yIGsgaW4gKG1heCgxLCByb3VuZChwb29sX2tlcm5lbF91bSAvIHMpKSBmb3IgcyBpbiB2b3hlbF9zaXplKSkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgayA9IG1heCgxLCByb3VuZChwb29sX2tlcm5lbF91bSkpOyBrZXJuID0gKGsgaWYgayAlIDIgPT0gMSBlbHNlIGsgKyAxLCkgKiAzCiAgICAgICAgICAgICAgICBwb29sZWQgPSBGLm1heF9wb29sM2QoZGV0X2xvZ2l0cywga2Vybiwgc3RyaWRlPTEsIHBhZGRpbmc9dHVwbGUoayAvLyAyIGZvciBrIGluIGtlcm4pKQogICAgICAgICAgICAgICAgaXNfcGVhayA9IChkZXRfbG9naXRzID09IHBvb2xlZCkgJiAoZGV0X2xvZ2l0cyA+IGRldF90aHJlc2hvbGQpCiAgICAgICAgICAgICAgICBjb3VudHMgPSBpc19wZWFrLmZsYXR0ZW4oMSkuc3VtKDEpCiAgICAgICAgICAgICAgICBpZiBib29sKChjb3VudHMgPiBjYXApLmFueSgpKToKICAgICAgICAgICAgICAgICAgICBkZXRfbG9naXRzID0gZGV0X2xvZ2l0cy5jbG9uZSgpCiAgICAgICAgICAgICAgICAgICAgZm9yIGIgaW4gdG9yY2gubm9uemVybyhjb3VudHMgPiBjYXApLmZsYXR0ZW4oKS50b2xpc3QoKToKICAgICAgICAgICAgICAgICAgICAgICAgdGhyX2IgPSB0b3JjaC50b3BrKGRldF9sb2dpdHNbYl1baXNfcGVha1tiXV0sIGNhcCkudmFsdWVzLm1pbigpCiAgICAgICAgICAgICAgICAgICAgICAgIGRldF9sb2dpdHNbYl0gPSB0b3JjaC53aGVyZShkZXRfbG9naXRzW2JdID49IHRocl9iLCBkZXRfbG9naXRzW2JdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdG9yY2guZnVsbF9saWtlKGRldF9sb2dpdHNbYl0sIGRldF90aHJlc2hvbGQgLSAxLjApKQogICAgICAgICAgICAgICAgICAgIENBUFsibl9jYXBwZWQiXSArPSAxCiAgICAgICAgcmV0dXJuIF9vcmlnX2RtKGRldF9sb2dpdHMsIGd0X2Nvb3JkcywgbWFzaywgaW1hZ2Vfc2hhcGUsIGRldF90aHJlc2hvbGQsIHBvb2xfa2VybmVsX3VtLCBtYXhfbWF0Y2hfZGlzdGFuY2UsCiAgICAgICAgICAgICAgICAgICAgICAgIHZveGVsX3NpemUsIGZyYW1lX2luZGV4LCB3aW5kb3dfc2l6ZSkKCiAgICBULmRldGVjdF9hbmRfbWF0Y2ggPSBkZXRlY3RfYW5kX21hdGNoX2NhcHBlZAogICAgX1QgPSBUCiAgICByZXR1cm4gVAoKCkNBUCA9IHsibWF4X2RldF9wZXJfZnJhbWUiOiAxMjAwLCAibl9jYXBwZWQiOiAwfSAgICMgc2V0IENBUFsibWF4X2RldF9wZXJfZnJhbWUiXSA9IE5vbmUgdG8gZGlzYWJsZQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHNwbGl0cyAoaWRlbnRpY2FsIHJ1bGUgdG8gb3VyIENWIC8gdjExIGhvbGQtb3V0KQpkZWYgbWFrZV9zcGxpdHModHJhaW5fZGlyOiBQYXRoLCBuX2hvbGRvdXQ6IGludCA9IDEyKSAtPiBkaWN0OgogICAgc3RlbXMgPSBzb3J0ZWQocC5uYW1lWzotNV0gZm9yIHAgaW4gUGF0aCh0cmFpbl9kaXIpLml0ZXJkaXIoKQogICAgICAgICAgICAgICAgICAgaWYgcC5uYW1lLmVuZHN3aXRoKCIuemFyciIpIGFuZCAoUGF0aCh0cmFpbl9kaXIpIC8gKHAubmFtZVs6LTVdICsgIi5nZWZmIikpLmV4aXN0cygpKQogICAgc3RlcCA9IG1heCgxLCBsZW4oc3RlbXMpIC8vIG1heCgxLCBuX2hvbGRvdXQpKQogICAgaG9sZCA9IHN0ZW1zWzE6OnN0ZXBdWzpuX2hvbGRvdXRdCiAgICBocyA9IHNldChob2xkKQogICAgcmV0dXJuIHsic3BsaXQiOiAwLCAidHJhaW4iOiBbcyBmb3IgcyBpbiBzdGVtcyBpZiBzIG5vdCBpbiBoc10sICJ0ZXN0IjogaG9sZH0KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBkYXRhCmRlZiBidWlsZF9sb2FkZXJzKGRhdGFfZGlyOiBQYXRoLCB0cmFpbl9zdGVtcywgdGVzdF9zdGVtcywgKiwgZG93bnNhbXBsZT0oMSwgNCwgNCksIHdpbmRvd19zaXplPTIsIGJhdGNoX3NpemU9MTYsCiAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPTQsIG1heF9mcmFtZXM9Tm9uZSwgYXVnbWVudD1UcnVlLCBsb2c9cHJpbnQpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKICAgIFQgPSBfVAoKICAgIGRlZiBfbG9hZChzdGVtcywgZGVzYyk6CiAgICAgICAgZGF0YSA9IFtdCiAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgIGZvciBpLCBzIGluIGVudW1lcmF0ZShzdGVtcyk6CiAgICAgICAgICAgIHZtLCB3aW5kb3dzID0gVC5sb2FkX2RhdGFzZXRfd2luZG93cyhQYXRoKGRhdGFfZGlyKSAvIHMsIHdpbmRvd19zaXplPXdpbmRvd19zaXplLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X2ZyYW1lcz1tYXhfZnJhbWVzLCBkb3duc2FtcGxlPWRvd25zYW1wbGUpCiAgICAgICAgICAgIGRhdGEuYXBwZW5kKCh2bSwgd2luZG93cykpCiAgICAgICAgICAgIGlmIChpICsgMSkgJSAyNSA9PSAwIG9yIGkgKyAxID09IGxlbihzdGVtcyk6CiAgICAgICAgICAgICAgICBsb2coZiIgICAge2Rlc2N9OiB7aSsxfS97bGVuKHN0ZW1zKX0gY2xpcHMsIHtzdW0obGVuKHcpIGZvciBfLCB3IGluIGRhdGEpfSB3aW5kb3dzICh7dGltZS50aW1lKCktdDA6LjBmfXMpIikKICAgICAgICByZXR1cm4gZGF0YQoKICAgIHRyID0gX2xvYWQodHJhaW5fc3RlbXMsICJ0cmFpbiIpOyB0ZSA9IF9sb2FkKHRlc3Rfc3RlbXMsICJ2YWwiKQogICAgYWxsX3dpbmRvd3MgPSBbdyBmb3IgXywgd3MgaW4gdHIgKyB0ZSBmb3IgdyBpbiB3c10KICAgIG1heF9ub2RlcyA9IG1heChtYXgody5ub2RlX2NvdW50cykgZm9yIHcgaW4gYWxsX3dpbmRvd3MpCiAgICBsb2coZiIgICAgbWF4IEdUIG5vZGVzIHBlciBmcmFtZSA9IHttYXhfbm9kZXN9IikKICAgIHRyYWluX2RzID0gVC5GcmFtZVdpbmRvd0RhdGFzZXQodHIsIG1heF9ub2Rlcz1tYXhfbm9kZXMsIGF1Z21lbnRhdGlvbnM9VC5ERUZBVUxUX0FVR01FTlRBVElPTlMgaWYgYXVnbWVudCBlbHNlIE5vbmUpCiAgICB2YWxfZHMgPSBULkZyYW1lV2luZG93RGF0YXNldCh0ZSwgbWF4X25vZGVzPW1heF9ub2RlcykKICAgIGt3ID0gZGljdChudW1fd29ya2Vycz1udW1fd29ya2VycywgcHJlZmV0Y2hfZmFjdG9yPTIgaWYgbnVtX3dvcmtlcnMgPiAwIGVsc2UgTm9uZSwKICAgICAgICAgICAgICBwZXJzaXN0ZW50X3dvcmtlcnM9bnVtX3dvcmtlcnMgPiAwLCBwaW5fbWVtb3J5PUZhbHNlKQogICAgdHJhaW5fbG9hZGVyID0gRGF0YUxvYWRlcih0cmFpbl9kcywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBzaHVmZmxlPVRydWUsICoqa3cpCiAgICB2YWxfbG9hZGVyID0gRGF0YUxvYWRlcih2YWxfZHMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSwgc2h1ZmZsZT1GYWxzZSwgKiprdykKICAgIHJldHVybiB0cmFpbl9sb2FkZXIsIHZhbF9sb2FkZXIsIG1heF9ub2RlcwoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG1vZGVsIC8gY2hlY2twb2ludHMKZGVmIGJ1aWxkX21vZGVsKGNmZzogZGljdCwgZGV2aWNlLCBkYXRhX3BhcmFsbGVsPVRydWUsIGxvZz1wcmludCk6CiAgICBpbXBvcnQgdG9yY2gKICAgIGltcG9ydCB0b3JjaC5ubiBhcyBubgogICAgVCA9IF9UCiAgICB1bmV0ID0gVC5UZW1wb3JhbFVOZXQzRChpbl9jaGFubmVscz0xLCBvdXRfY2hhbm5lbHM9Y2ZnWyJ1bmV0X291dF9jaGFubmVscyJdLCBsYXllcnM9Y2ZnWyJ1bmV0X2xheWVycyJdKQogICAgbW9kZWwgPSBULlVOZXROb2RlVHJhbnNmb3JtZXIodW5ldD11bmV0LCB1bmV0X291dF9jaGFubmVscz1jZmdbInVuZXRfb3V0X2NoYW5uZWxzIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwb3NfZmVhdF9kaW09NCAqIFQuX1BPU19FTUJFRF9ESU0pLnRvKGRldmljZSkKICAgIG5fZ3B1ID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSBpZiBkZXZpY2UudHlwZSA9PSAiY3VkYSIgZWxzZSAwCiAgICBpZiBkYXRhX3BhcmFsbGVsIGFuZCBuX2dwdSA+IDE6CiAgICAgICAgbW9kZWwudW5ldCA9IG5uLkRhdGFQYXJhbGxlbChtb2RlbC51bmV0KQogICAgICAgIGxvZyhmIiAgICBEYXRhUGFyYWxsZWw6IFVOZXQgYWNyb3NzIHtuX2dwdX0gR1BVcyIpCiAgICBsb2coZiIgICAgcGFyYW1ldGVyczoge3N1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKTosfSIpCiAgICByZXR1cm4gbW9kZWwKCgpkZWYgX3BsYWluX3N0YXRlKG1vZGVsKToKICAgICIiInN0YXRlX2RpY3Qgd2l0aCB0aGUgRGF0YVBhcmFsbGVsIHByZWZpeCByZW1vdmVkICh0aGUgcmVwbydzIGNoZWNrcG9pbnQgZm9ybWF0KS4iIiIKICAgIHJldHVybiB7ay5yZXBsYWNlKCJ1bmV0Lm1vZHVsZS4iLCAidW5ldC4iLCAxKTogdi5kZXRhY2goKS5jcHUoKSBmb3IgaywgdiBpbiBtb2RlbC5zdGF0ZV9kaWN0KCkuaXRlbXMoKX0KCgpkZWYgX2xvYWRfcGxhaW5fc3RhdGUobW9kZWwsIHN0YXRlKToKICAgIGltcG9ydCB0b3JjaC5ubiBhcyBubgogICAgaWYgaXNpbnN0YW5jZShtb2RlbC51bmV0LCBubi5EYXRhUGFyYWxsZWwpOgogICAgICAgIHN0YXRlID0geyhrLnJlcGxhY2UoInVuZXQuIiwgInVuZXQubW9kdWxlLiIsIDEpIGlmIGsuc3RhcnRzd2l0aCgidW5ldC4iKSBlbHNlIGspOiB2IGZvciBrLCB2IGluIHN0YXRlLml0ZW1zKCl9CiAgICBtaXNzaW5nLCB1bmV4cGVjdGVkID0gbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHN0YXRlLCBzdHJpY3Q9RmFsc2UpCiAgICByZXR1cm4gbWlzc2luZywgdW5leHBlY3RlZAoKCmRlZiBmaW5kX3Jlc3VtZShkaXJzLCBsb2c9cHJpbnQpOgogICAgIiIiTmV3ZXN0IGxhc3QucHRoIGFtb25nIGNhbmRpZGF0ZSB3ZWlnaHQgZGlycyAoYXR0YWNoZWQgb3V0cHV0cyBvZiBhIHByZXZpb3VzIHJ1bikuIiIiCiAgICBjYW5kcyA9IFtdCiAgICBmb3IgZCBpbiBkaXJzOgogICAgICAgIGZvciBwIGluIFBhdGgoZCkucmdsb2IoImxhc3QucHRoIik6CiAgICAgICAgICAgIGNhbmRzLmFwcGVuZChwKQogICAgaWYgbm90IGNhbmRzOgogICAgICAgIHJldHVybiBOb25lCiAgICBjYW5kcy5zb3J0KGtleT1sYW1iZGEgcDogcC5zdGF0KCkuc3RfbXRpbWUpCiAgICBsb2coZiIgICAgcmVzdW1lIGNoZWNrcG9pbnQ6IHtjYW5kc1stMV19IikKICAgIHJldHVybiBjYW5kc1stMV0KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSB0cmFpbmluZyBsb29wIHdpdGggYnVkZ2V0ICsgcmVzdW1lCmRlZiB0cmFpbl9sb29wKG1vZGVsLCB0cmFpbl9sb2FkZXIsIHZhbF9sb2FkZXIsICosIG91dF9kaXI6IFBhdGgsIGNmZzogZGljdCwgZXBvY2hzOiBpbnQsIGxyOiBmbG9hdCA9IDFlLTQsCiAgICAgICAgICAgICAgIGRldF9sb3NzX3dlaWdodD0xLjAsIGRldF9uZWdfd2VpZ2h0PTFlLTIsIHRpbWVfYnVkZ2V0X3NlYz0xMCAqIDM2MDAsIHJlc3VtZTogUGF0aCB8IE5vbmUgPSBOb25lLAogICAgICAgICAgICAgICBtYXhfaXRlcnM9Tm9uZSwgbHJfbWluX2ZyYWM9MC4xLCBsb2c9cHJpbnQpOgogICAgIiIiVHJhaW5zIHVudGlsIGBlcG9jaHNgIHRvdGFsIChjb3VudGluZyByZXN1bWVkIGVwb2Nocykgb3IgdGhlIHRpbWUgYnVkZ2V0IGlzIGV4aGF1c3RlZC4KICAgIFNhdmVzIGxhc3QucHRoIChmdWxsIHJlc3VtZSBzdGF0ZSksIGVkZ2VfcHJlZGljdG9yX2Jlc3QucHRoIC8gZWRnZV9wcmVkaWN0b3JfbGFzdC5wdGggKHJlcG8gZm9ybWF0KSwgY29uZmlnLmpzb24uIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIFQgPSBfVAogICAgZGV2aWNlID0gbmV4dChtb2RlbC5wYXJhbWV0ZXJzKCkpLmRldmljZQogICAgb3V0X2RpciA9IFBhdGgob3V0X2Rpcik7IG91dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgKG91dF9kaXIgLyAiY29uZmlnLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoCiAgICAgICAge2s6IGNmZ1trXSBmb3IgayBpbiAoInVuZXRfb3V0X2NoYW5uZWxzIiwgInVuZXRfbGF5ZXJzIiwgImRvd25zYW1wbGUiLCAid2luZG93X3NpemUiLCAicG9vbF9rZXJuZWxfdW0iKX0sIGluZGVudD0yKSkKICAgIG9wdCA9IHRvcmNoLm9wdGltLkFkYW1XKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9bHIpCiAgICBzdGFydF9lcG9jaCwgYmVzdCwgaGlzdCwgYmVzdF9tb2RlbF9zdGF0ZSA9IDAsIDAuMCwgW10sIE5vbmUKICAgIGlmIHJlc3VtZSBpcyBub3QgTm9uZSBhbmQgUGF0aChyZXN1bWUpLmV4aXN0cygpOgogICAgICAgIGNrID0gdG9yY2gubG9hZChyZXN1bWUsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgIG1pLCB1biA9IF9sb2FkX3BsYWluX3N0YXRlKG1vZGVsLCBja1sibW9kZWwiXSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIG9wdC5sb2FkX3N0YXRlX2RpY3QoY2tbIm9wdGltaXplciJdKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICMgbm9xYQogICAgICAgICAgICBsb2coZiIgICAgb3B0aW1pemVyIHN0YXRlIG5vdCByZXN0b3JlZCAoe3R5cGUoZSkuX19uYW1lX199KSAtIGZyZXNoIG9wdGltaXplciIpCiAgICAgICAgc3RhcnRfZXBvY2gsIGJlc3QsIGhpc3QgPSBpbnQoY2tbImVwb2NoIl0pLCBmbG9hdChjay5nZXQoImJlc3QiLCAwLjApKSwgbGlzdChjay5nZXQoImhpc3RvcnkiLCBbXSkpCiAgICAgICAgYmVzdF9tb2RlbF9zdGF0ZSA9IGNrLmdldCgiYmVzdF9tb2RlbCIpCiAgICAgICAgaWYgYmVzdF9tb2RlbF9zdGF0ZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgdG9yY2guc2F2ZShiZXN0X21vZGVsX3N0YXRlLCBvdXRfZGlyIC8gImVkZ2VfcHJlZGljdG9yX2Jlc3QucHRoIikgICAjIGNhcnJ5IHRoZSBiZXN0IHdlaWdodHMgaW50byB0aGlzIHJ1bidzIG91dHB1dAogICAgICAgIHRvcmNoLnNhdmUoY2ssIG91dF9kaXIgLyAibGFzdC5wdGgiKTsgdG9yY2guc2F2ZShja1sibW9kZWwiXSwgb3V0X2RpciAvICJlZGdlX3ByZWRpY3Rvcl9sYXN0LnB0aCIpICAgIyBvdXRwdXQgaXMgY29tcGxldGUgZXZlbiBpZiBubyBlcG9jaCBmaXRzCiAgICAgICAgKG91dF9kaXIgLyAiaGlzdG9yeS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKGhpc3QsIGluZGVudD0xKSkKICAgICAgICBsb2coZiIgICAgcmVzdW1lZCBhdCBlcG9jaCB7c3RhcnRfZXBvY2h9IChiZXN0IGFjYypyZWNhbGwgc28gZmFyIHtiZXN0Oi40Zn07IG1pc3Npbmc9e2xlbihtaSl9IHVuZXhwZWN0ZWQ9e2xlbih1bil9KSIpCgogICAgZGVmIGxyX2F0KGVwKTogICMgY29zaW5lIG92ZXIgdGhlIHBsYW5uZWQgdG90YWwsIGZsb29yIGF0IGxyX21pbl9mcmFjCiAgICAgICAgcmV0dXJuIGxyICogKGxyX21pbl9mcmFjICsgKDEgLSBscl9taW5fZnJhYykgKiAwLjUgKiAoMSArIG1hdGguY29zKG1hdGgucGkgKiBtaW4oMS4wLCBlcCAvIG1heCgxLCBlcG9jaHMpKSkpKQoKICAgIHRfc3RhcnQgPSB0aW1lLnRpbWUoKTsgZXBfdGltZXMgPSBbXQogICAgZm9yIGVwIGluIHJhbmdlKHN0YXJ0X2Vwb2NoLCBlcG9jaHMpOgogICAgICAgIGZvciBnIGluIG9wdC5wYXJhbV9ncm91cHM6CiAgICAgICAgICAgIGdbImxyIl0gPSBscl9hdChlcCkKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgZWRnZV9sb3NzLCBkZXRfbG9zcyA9IFQudHJhaW5fZXBvY2gobW9kZWwsIHRyYWluX2xvYWRlciwgb3B0LCBkZXZpY2UsIGRldF9sb3NzX3dlaWdodCwgZGV0X25lZ193ZWlnaHQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X2l0ZXJzPW1heF9pdGVycywgcG9vbF9rZXJuZWxfdW09Y2ZnWyJwb29sX2tlcm5lbF91bSJdKQogICAgICAgIHRfdHJhaW4gPSB0aW1lLnRpbWUoKSAtIHQwOyB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIHZhbF9sb3NzLCB2YWxfYWNjLCB2YWxfcmVjYWxsID0gVC5ldmFsdWF0ZShtb2RlbCwgdmFsX2xvYWRlciwgZGV2aWNlLCBwb29sX2tlcm5lbF91bT1jZmdbInBvb2xfa2VybmVsX3VtIl0pCiAgICAgICAgdF92YWwgPSB0aW1lLnRpbWUoKSAtIHQwCiAgICAgICAgc2NvcmUgPSB2YWxfYWNjICogdmFsX3JlY2FsbAogICAgICAgIGlzX2Jlc3QgPSBzY29yZSA+PSBiZXN0CiAgICAgICAgc3RhdGUgPSBfcGxhaW5fc3RhdGUobW9kZWwpCiAgICAgICAgaWYgaXNfYmVzdDoKICAgICAgICAgICAgYmVzdCA9IHNjb3JlOyBiZXN0X21vZGVsX3N0YXRlID0gc3RhdGUKICAgICAgICAgICAgdG9yY2guc2F2ZShzdGF0ZSwgb3V0X2RpciAvICJlZGdlX3ByZWRpY3Rvcl9iZXN0LnB0aCIpCiAgICAgICAgdG9yY2guc2F2ZShzdGF0ZSwgb3V0X2RpciAvICJlZGdlX3ByZWRpY3Rvcl9sYXN0LnB0aCIpCiAgICAgICAgcm93ID0gZGljdChlcG9jaD1lcCArIDEsIGxyPWxyX2F0KGVwKSwgZWRnZV9sb3NzPWVkZ2VfbG9zcywgZGV0X2xvc3M9ZGV0X2xvc3MsIHZhbF9sb3NzPXZhbF9sb3NzLAogICAgICAgICAgICAgICAgICAgdmFsX2FjYz12YWxfYWNjLCB2YWxfcmVjYWxsPXZhbF9yZWNhbGwsIHNjb3JlPXNjb3JlLCBiZXN0PWJlc3QsIHRfdHJhaW49dF90cmFpbiwgdF92YWw9dF92YWwpCiAgICAgICAgaGlzdC5hcHBlbmQocm93KQogICAgICAgIHRvcmNoLnNhdmUoZGljdChtb2RlbD1zdGF0ZSwgb3B0aW1pemVyPW9wdC5zdGF0ZV9kaWN0KCksIGVwb2NoPWVwICsgMSwgYmVzdD1iZXN0LCBoaXN0b3J5PWhpc3QsCiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3RfbW9kZWw9YmVzdF9tb2RlbF9zdGF0ZSwgY2ZnPWNmZyksIG91dF9kaXIgLyAibGFzdC5wdGgiKQogICAgICAgIChvdXRfZGlyIC8gImhpc3RvcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhoaXN0LCBpbmRlbnQ9MSkpCiAgICAgICAgbG9nKGYiICBlcG9jaCB7ZXArMTozZH0ve2Vwb2Noc30gbHIge2xyX2F0KGVwKTouMWV9IHwgZWRnZSB7ZWRnZV9sb3NzOi40Zn0gZGV0IHtkZXRfbG9zczouNGZ9IHwgIgogICAgICAgICAgICBmInZhbCBsb3NzIHt2YWxfbG9zczouNGZ9IGFjYyB7dmFsX2FjYzouNGZ9IHJlY2FsbCB7dmFsX3JlY2FsbDouNGZ9IC0+IHtzY29yZTouNGZ9IHsnKicgaWYgaXNfYmVzdCBlbHNlICcnfSAiCiAgICAgICAgICAgIGYiKGJlc3Qge2Jlc3Q6LjRmfSkgfCB0cmFpbiB7dF90cmFpbjouMGZ9cyB2YWwge3RfdmFsOi4wZn1zIHwgZGV0LWNhcCBoaXRzIHtDQVBbJ25fY2FwcGVkJ119IikKICAgICAgICBDQVBbIm5fY2FwcGVkIl0gPSAwCiAgICAgICAgZXBfdGltZXMuYXBwZW5kKHRfdHJhaW4gKyB0X3ZhbCkKICAgICAgICBlbGFwc2VkID0gdGltZS50aW1lKCkgLSB0X3N0YXJ0CiAgICAgICAgaWYgZXAgKyAxIDwgZXBvY2hzIGFuZCBlbGFwc2VkICsgMS4xNSAqIG5wLm1lYW4oZXBfdGltZXMpID4gdGltZV9idWRnZXRfc2VjOgogICAgICAgICAgICBsb2coZiIgIHRpbWUgYnVkZ2V0OiB7ZWxhcHNlZC8zNjAwOi4yZn0gaCB1c2VkLCBuZXh0IGVwb2NoICh+e25wLm1lYW4oZXBfdGltZXMpLzYwOi4wZn0gbWluKSB3b3VsZCBub3QgZml0IC0+IHN0b3BwaW5nIGF0IGVwb2NoIHtlcCsxfSIpCiAgICAgICAgICAgIGJyZWFrCiAgICByZXR1cm4gaGlzdCwgYmVzdAo="),
                  ("cellmot_infer.py", "IiIiSW5mZXJlbmNlIGhlbHBlcnMgYXJvdW5kIHRoZSByZXBvJ3MgcHJlZGljdF91bmV0X3RyYW5zZm9ybWVyLnB5OiBjb29yZHMvZWRnZXMgLT4gb3VyIG5vZGUvZWRnZSBsaXN0cywKb3B0aW9uYWwgdHJhY2tzZGF0YSBJTFAsIHNjb3Jpbmcgd2l0aCBvdXIgb2ZmaWNpYWwtc3R5bGUgc2NvcmVyLCBzdWJtaXNzaW9uIHJvd3MuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKaW1wb3J0IHN5cywgdGltZSwgY29udGV4dGxpYiwgaW8KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucAoKX1AgPSBOb25lCgoKZGVmIHNldHVwX3ByZWRpY3QocmVwb19kaXI6IFBhdGgpOgogICAgZ2xvYmFsIF9QCiAgICBpbXBvcnQgY2VsbG1vdF90cmFpbiBhcyBDVAogICAgQ1Quc2V0dXBfcmVwbyhyZXBvX2RpcikKICAgIGltcG9ydCBwcmVkaWN0X3VuZXRfdHJhbnNmb3JtZXIgYXMgUAogICAgaW1wb3J0IHRyYWNraW5nX2NlbGxtb3QuaW8gYXMgY2lvCiAgICBQLm9wZW5fZGF0YXNldCA9IGNpby5vcGVuX2RhdGFzZXQgICAgICAgICAgIyBxdWFudGlsZSBmYWxsYmFjayBwYXRjaAogICAgX1AgPSBQCiAgICByZXR1cm4gUAoKCmRlZiBtYWtlX2NmZyhkZXRfdGhyZXNob2xkPTAuOTUsIHVzZV9pbHA9VHJ1ZSwgaWxwX2VkZ2Vfd2VpZ2h0PS0xLjAsIGlscF9hcHBlYXJhbmNlX3dlaWdodD0wLjAsCiAgICAgICAgICAgICBpbHBfZGlzYXBwZWFyYW5jZV93ZWlnaHQ9MS40LCBpbHBfZGl2aXNpb25fd2VpZ2h0PTEuMCwgZWRnZV90aHJlc2hvbGQ9MC4yNSwgcG9vbF9rZXJuZWxfdW09My4wLCBkZXRfdHRhPVRydWUpOgogICAgcmV0dXJuIF9QLlByZWRpY3RDb25maWcoZGV0X3RocmVzaG9sZD1kZXRfdGhyZXNob2xkLCBkZXRfdHRhPWRldF90dGEsIHBvb2xfa2VybmVsX3VtPXBvb2xfa2VybmVsX3VtLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgdGhyZXNob2xkPWVkZ2VfdGhyZXNob2xkLCB1c2VfaWxwPXVzZV9pbHAsIGlscF9lZGdlX3dlaWdodD1pbHBfZWRnZV93ZWlnaHQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBpbHBfYXBwZWFyYW5jZV93ZWlnaHQ9aWxwX2FwcGVhcmFuY2Vfd2VpZ2h0LCBpbHBfZGlzYXBwZWFyYW5jZV93ZWlnaHQ9aWxwX2Rpc2FwcGVhcmFuY2Vfd2VpZ2h0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgaWxwX2RpdmlzaW9uX3dlaWdodD1pbHBfZGl2aXNpb25fd2VpZ2h0KQoKCmRlZiBsb2FkX21vZGVsKHdlaWdodHNfcGF0aDogUGF0aCwgZGV2aWNlKToKICAgIHJldHVybiBfUC5sb2FkX21vZGVsKFBhdGgod2VpZ2h0c19wYXRoKSwgZGV2aWNlKQoKCmRlZiBwcmVkaWN0X2NsaXAobW9kZWwsIHphcnJfcGF0aDogUGF0aCwgZGV2aWNlLCBjZmcsIHdpbmRvd19zaXplLCBkb3duc2FtcGxlLCBsb2c9cHJpbnQpOgogICAgIiIiUmV0dXJucyAobm9kZXMgWyh0LHoseSx4LGlkKV0sIGVkZ2VzIFsoc3JjX2lkLHRndF9pZCldLCB3ZWlnaHRzKSBpbiBPVVIgZm9ybWF0IChpZHMgc3RhcnQgYXQgMSkuIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIHQwID0gdGltZS50aW1lKCkKICAgIGNvb3JkcywgZWRnZXMgPSBfUC5wcmVkaWN0X3ZpZGVvKG1vZGVsLCBQYXRoKHphcnJfcGF0aCksIGRldmljZSwgY2ZnPWNmZywgd2luZG93X3NpemU9d2luZG93X3NpemUsIGRvd25zYW1wbGU9ZG93bnNhbXBsZSkKICAgIG4gPSBsZW4oY29vcmRzKQogICAgaWYgY2ZnLnVzZV9pbHAgYW5kIGVkZ2VzOgogICAgICAgIGltcG9ydCB0cmFja3NkYXRhIGFzIHRkCiAgICAgICAgZ3JhcGggPSBfUC5idWlsZF9ncmFwaChjb29yZHMsIGVkZ2VzKQogICAgICAgIHNvbHZlciA9IHRkLnNvbHZlcnMuSUxQU29sdmVyKGVkZ2Vfd2VpZ2h0PWNmZy5pbHBfZWRnZV93ZWlnaHQgKiB0ZC5FZGdlQXR0cigiZWRnZV9wcm9iIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYXBwZWFyYW5jZV93ZWlnaHQ9Y2ZnLmlscF9hcHBlYXJhbmNlX3dlaWdodCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkaXNhcHBlYXJhbmNlX3dlaWdodD1jZmcuaWxwX2Rpc2FwcGVhcmFuY2Vfd2VpZ2h0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRpdmlzaW9uX3dlaWdodD1jZmcuaWxwX2RpdmlzaW9uX3dlaWdodCkKICAgICAgICB3aXRoIGNvbnRleHRsaWIucmVkaXJlY3Rfc3Rkb3V0KGlvLlN0cmluZ0lPKCkpLCBjb250ZXh0bGliLnJlZGlyZWN0X3N0ZGVycihpby5TdHJpbmdJTygpKToKICAgICAgICAgICAgc29sdmVyLnNvbHZlKGdyYXBoKSAgICAgICAgICAgICAgICAgICAgICAgIyB3cml0ZXMgYSBib29sZWFuICJzb2x1dGlvbiIgYXR0ciBvbiBub2RlcyBhbmQgZWRnZXMKICAgICAgICBuYSA9IGdyYXBoLm5vZGVfYXR0cnMoYXR0cl9rZXlzPVsibm9kZV9pZCIsICJ0IiwgInoiLCAieSIsICJ4IiwgInNvbHV0aW9uIl0pCiAgICAgICAgZWEgPSBncmFwaC5lZGdlX2F0dHJzKGF0dHJfa2V5cz1bInNvdXJjZV9pZCIsICJ0YXJnZXRfaWQiLCAic29sdXRpb24iXSkKICAgICAgICBuYSA9IG5hLmZpbHRlcihuYVsic29sdXRpb24iXSk7IGVhID0gZWEuZmlsdGVyKGVhWyJzb2x1dGlvbiJdKQogICAgICAgIGlmIGxlbihuYSkgPT0gMDogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGRlZ2VuZXJhdGUgc29sdXRpb24gLT4ga2VlcCB0aGUgcmF3IGRldGVjdGlvbnMKICAgICAgICAgICAgbmEgPSBncmFwaC5ub2RlX2F0dHJzKGF0dHJfa2V5cz1bIm5vZGVfaWQiLCAidCIsICJ6IiwgInkiLCAieCJdKQogICAgICAgIGlkMm5ldyA9IHtpbnQoaSk6IGsgKyAxIGZvciBrLCBpIGluIGVudW1lcmF0ZShuYVsibm9kZV9pZCJdLnRvX2xpc3QoKSl9CiAgICAgICAgbm9kZXMgPSBbKGludCh0KSwgaW50KHJvdW5kKHopKSwgaW50KHJvdW5kKHkpKSwgaW50KHJvdW5kKHgpKSwgaWQybmV3W2ludChpKV0pCiAgICAgICAgICAgICAgICAgZm9yIGksIHQsIHosIHksIHggaW4gemlwKG5hWyJub2RlX2lkIl0udG9fbGlzdCgpLCBuYVsidCJdLnRvX2xpc3QoKSwgbmFbInoiXS50b19saXN0KCksIG5hWyJ5Il0udG9fbGlzdCgpLCBuYVsieCJdLnRvX2xpc3QoKSldCiAgICAgICAgb3V0X2VkZ2VzID0gWyhpZDJuZXdbaW50KHMpXSwgaWQybmV3W2ludCh0KV0pIGZvciBzLCB0IGluIHppcChlYVsic291cmNlX2lkIl0udG9fbGlzdCgpLCBlYVsidGFyZ2V0X2lkIl0udG9fbGlzdCgpKQogICAgICAgICAgICAgICAgICAgICBpZiBpbnQocykgaW4gaWQybmV3IGFuZCBpbnQodCkgaW4gaWQybmV3XQogICAgICAgIHdlaWdodHMgPSBbMS4wXSAqIGxlbihvdXRfZWRnZXMpCiAgICBlbHNlOgogICAgICAgIG5vZGVzID0gWyhpbnQodCksIGludCh6KSwgaW50KHkpLCBpbnQoeCksIGsgKyAxKSBmb3IgaywgKHQsIHosIHksIHgpIGluIGVudW1lcmF0ZShjb29yZHMudG9saXN0KCkpXQogICAgICAgIG91dF9lZGdlcyA9IFsoaW50KHMpICsgMSwgaW50KHQpICsgMSkgZm9yIHMsIHQsIHAsIGQgaW4gZWRnZXNdCiAgICAgICAgd2VpZ2h0cyA9IFtmbG9hdChwKSBmb3IgcywgdCwgcCwgZCBpbiBlZGdlc10KICAgIGxvZyhmIiAgICBjZWxsbW90OiB7bn0gZGV0ZWN0aW9ucywge2xlbihlZGdlcyl9IGNhbmRpZGF0ZSBlZGdlcyAtPiB7bGVuKG91dF9lZGdlcyl9IGVkZ2VzICh7dGltZS50aW1lKCktdDA6LjBmfXMpIikKICAgIHJldHVybiBub2Rlcywgb3V0X2VkZ2VzLCB3ZWlnaHRzCg==")]:
    (CODE_DIR / _name).write_bytes(base64.b64decode(_b))
sys.path.insert(0, str(CODE_DIR))
from pipeline import read_geff, score_official, micro_average, rows_for_sample, write_submission
import cellmot_infer as CI
assert REPO_DIR is not None, "official baseline repo not found: attach the prep-bundle v2 Output"
P = CI.setup_predict(REPO_DIR)

cands = [d / WEIGHTS_PREFER for d in CKPT_DIRS if (d / WEIGHTS_PREFER).exists()]
assert cands, f"no {WEIGHTS_PREFER} found in inputs: attach the v13 training Output"
cands.sort(key=lambda p: p.stat().st_mtime)
WEIGHTS = cands[-1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 4); print("CPU mode, threads:", torch.get_num_threads(), "| det_tta:", DET_TTA)
model, WINDOW, DOWNSAMPLE = CI.load_model(WEIGHTS, device)
hist_p = WEIGHTS.parent / "history.json"
if hist_p.exists():
    _h = json.loads(hist_p.read_text()); print(f"weights: {WEIGHTS} | trained epochs: {len(_h)} | best val acc*recall {max(r['score'] for r in _h):.4f}")
else:
    print("weights:", WEIGHTS)
print("window", WINDOW, "downsample", DOWNSAMPLE, "device", device)

def run_clip(zp, over, log=print):
    cfg = CI.make_cfg(**{"det_tta": DET_TTA, **over})
    nodes, edges, w = CI.predict_clip(model, zp, device, cfg, WINDOW, DOWNSAMPLE, log=log)
    return nodes, edges


In [ ]:
# ------------------------------------------------------------------ 4. CV on the 12 hold-out clips (official-style scorer)
if RUN_CV and (COMP_DIR / "train").is_dir():
    cv_zarrs = sorted(p for p in (COMP_DIR / "train").iterdir() if p.name.endswith(".zarr"))
    cv_zarrs = [p for p in cv_zarrs if (p.parent / (p.name[:-5] + ".geff")).exists()]
    step = max(1, len(cv_zarrs) // max(1, N_CV_SAMPLES))
    cv_zarrs = cv_zarrs[1::step][:N_CV_SAMPLES]
    print("CV clips:", [p.name[:-5] for p in cv_zarrs])
    gts = {zp: read_geff(zp.parent / (zp.name[:-5] + ".geff")) for zp in cv_zarrs}
    cv_rows, cv_summary = [], []
    for name, over in CV_CONFIGS.items():
        rows = []
        for zp in cv_zarrs:
            if time.time() - T_START > TIME_BUDGET_SEC:
                print("!! time budget exhausted during CV; remaining clips skipped"); break
            t_clip = time.time()
            nodes, edges = run_clip(zp, over, log=lambda *a: None)
            sc = score_official(nodes, edges, gts[zp]); sc.update(config=name, sample=zp.name[:-5]); rows.append(sc)
            print(f"    [{name}] {zp.name[:-5]}: score={sc['score']:.3f} (tp={sc['edge_tp']} fp={sc['edge_fp']} fn={sc['edge_fn']}) "
                  f"node_recall={sc['node_recall']:.3f} divJ={sc['div_J']:.2f} n_pred/est={sc['n_pred']}/{sc['est']} ({time.time()-t_clip:.0f}s)")
        if not rows: break
        ma = micro_average(rows); ma["config"] = name; cv_summary.append(ma); cv_rows += rows
        print(f"  ==> [{name}] micro score={ma['score']:.4f} adjJ={ma['adj_edge_J']:.4f} divJ={ma['div_J']:.3f}  ({time.time()-T_START:.0f}s)")
    import pandas as pd
    display(pd.DataFrame(cv_summary).set_index("config"))
    display(pd.DataFrame(cv_rows).pivot_table(index="sample", columns="config", values="score").round(3))
    pd.DataFrame(cv_rows).to_csv(OUT_DIR / "cv_rows.csv", index=False)
print("FINAL_CONFIG:", FINAL_CONFIG, "->", CV_CONFIGS[FINAL_CONFIG])
if MAX_TEST_SAMPLES == 0:
    print("MAX_TEST_SAMPLES = 0 -> CV only, no submission.csv (skip the next two cells)")


In [ ]:
# ------------------------------------------------------------------ 5. inference on test -> submission.csv
all_rows, summary = [], []
zarrs = TEST_ZARRS if MAX_TEST_SAMPLES is None else TEST_ZARRS[:MAX_TEST_SAMPLES]
sub_path = OUT_DIR / "submission.csv"
for k, zp in enumerate(zarrs):
    name = zp.name[:-5]
    if time.time() - T_START > TIME_BUDGET_SEC:
        print(f"!! time budget exhausted after {k} samples; remaining samples get a placeholder node")
        all_rows.append((name, "node", 1, 0, 0, 0, 0, -1, -1)); continue
    print(f"[{name}]")
    nodes, edges = run_clip(zp, CV_CONFIGS[FINAL_CONFIG])
    rows = rows_for_sample(name, nodes, edges) or [(name, "node", 1, 0, 0, 0, 0, -1, -1)]
    all_rows += rows; summary.append(dict(name=name, n_nodes=len(nodes), n_edges=len(edges)))
    print(f"    [{k+1}/{len(zarrs)}] done, elapsed {time.time()-T_START:.0f}s")
if zarrs:
    df = write_submission(all_rows, sub_path)
    print(df.row_type.value_counts().to_dict(), "| written:", sub_path, f"{sub_path.stat().st_size/1e6:.1f} MB")
else:
    print("no test inference requested (MAX_TEST_SAMPLES = 0)")


In [ ]:
# ------------------------------------------------------------------ 6. format self-check
import pandas as pd
if not zarrs:
    print("CV-only run: nothing to check")
else:
    sample = pd.read_csv(COMP_DIR / "sample_submission.csv")
    sub = pd.read_csv(sub_path)
    assert list(sub.columns) == list(sample.columns), (list(sub.columns), list(sample.columns))
    assert sub["id"].is_unique and (sub["id"] == range(len(sub))).all()
    nodes_ = sub[sub.row_type == "node"]; edges_ = sub[sub.row_type == "edge"]
    assert not nodes_.duplicated(["dataset", "node_id"]).any(), "duplicate node ids within a dataset"
    nid = set(zip(nodes_.dataset, nodes_.node_id))
    bad = [(d, s, t) for d, s, t in zip(edges_.dataset, edges_.source_id, edges_.target_id) if (d, s) not in nid or (d, t) not in nid]
    assert not bad, f"{len(bad)} edges reference unknown nodes, e.g. {bad[:3]}"
    assert set(p.name[:-5] for p in zarrs) <= set(sub.dataset.unique()), "some test datasets have no rows"
    for c in ["node_id", "t", "z", "y", "x", "source_id", "target_id"]:
        assert np.issubdtype(sub[c].dtype, np.integer), c
    print("format OK ✓ |", len(nodes_), "nodes,", len(edges_), "edges over", sub.dataset.nunique(), "datasets |", f"total {time.time()-T_START:.0f}s")
